# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (Local)

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Setup (same warehouse path as w04) ---
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass

    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

## 1. Method choice and why

**Lane 4 — CTR / Engagement Opportunity Scoring**  

**Method we will use:** Logistic Regression → rank pages by predicted probability of `is_ctr_underperformer`.

**Why:**
- We need a **ranked review list**, not just yes/no.
- We already have a clear label: CTR below the tier median with `imp_mar >= 500`.
- Logistic regression is **simple and readable** — we can explain what drives the score.
- Week 4’s baseline ranks the **`imp_mar >= 500` pool** by `ctr_gap` (with `imp_mar` tie-break). That gives perfect Precision@K by design, but it **floods the top** with zero-click `top_3` ties. A model can also use `imp_mar`, `pos_avg_mar`, and `engagement_rate_mar` to put **high-stake non-zero CTR underperformers** first.

**Not using yet:** Random Forest or boosting — only if logistic regression clearly adds value on a held-out client split.

**Not as features:** `ctr_gap`, tier median, or IDs (leakage / not signals — see ML-04).

## 2. Split design

**Split: client holdout (grouped by `client_hash_id`)**

- **Train:** ~75% of clients (all their March pages)
- **Test:** ~25% of clients held out — model never saw these clients during training
- **Same slice as Week 4:** March 2026 only (`month=2026-03`), `imp_mar >= 100` for features, real position only

**Why client holdout?**  
Pages from the same client share site patterns (niche, CMS, tracking). A random page split would put similar pages in both train and test — the score would look too good.

**Why not time-aware?**  
We are not predicting the future. Features and label both come from March 2026 — one snapshot, “which pages to review now?” Time splits are for later capstone work.

**Fair comparison:**  
Baseline (`ctr_gap`) ranks the **`imp_mar >= 500` test pages** (same pool as ML-07). Model scores all test pages but we compare Precision@K on that **eligible test subset** so both methods face the same label base rate.

**Comparison contract (rule = model):**
- same rows: eligible holdout pages (`imp_mar >= 500`)
- same holdout: same test clients
- same metric + K: Precision@10 / @20 / @50
- same tie policy: higher score first, then higher `imp_mar`


## 3. Train + compare vs my baseline

In [2]:
LABEL = "is_ctr_underperformer"
FEATURE_COLS = ["imp_mar", "ctr_mar", "pos_avg_mar", "engagement_rate_mar", "position_tier"]
RANDOM_STATE = 42
IMP_FLOOR = 500


def precision_at_k(scores, labels, k, tie_break=None):
    """Rank by score desc, then tie_break desc (same contract for rule and model)."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels)
    if tie_break is None:
        order = np.argsort(-scores, kind="mergesort")
    else:
        tb = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tb, -scores))  # last key is primary
    k = min(k, len(order))
    return float(labels[order[:k]].mean())


# --- Same March slice + honest features as ML-04 / ML-03 ---
features = con.sql(
    f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_mar,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN sessions_mar > 0 THEN 100.0 * engaged_mar / sessions_mar
            END AS engagement_rate_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier)
                     AND imp_mar >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
"""
).df()

model_df = features.dropna(subset=["ctr_mar", "pos_avg_mar"]).copy()
model_df["engagement_rate_mar"] = model_df["engagement_rate_mar"].fillna(0)
model_df["baseline_score"] = model_df["tier_median_ctr"] - model_df["ctr_mar"]

# --- Client holdout: ~75% train / ~25% test ---
rng = np.random.default_rng(RANDOM_STATE)
clients = np.sort(model_df["client_hash_id"].unique())
rng.shuffle(clients)
n_test_clients = max(1, int(round(len(clients) * 0.25)))
test_clients = set(clients[:n_test_clients])

train_df = model_df[~model_df["client_hash_id"].isin(test_clients)].copy()
test_df = model_df[model_df["client_hash_id"].isin(test_clients)].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df[LABEL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[LABEL].to_numpy()

preprocessor = ColumnTransformer(
    [
        (
            "num",
            StandardScaler(),
            ["imp_mar", "ctr_mar", "pos_avg_mar", "engagement_rate_mar"],
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["position_tier"]),
    ]
)

model = Pipeline(
    [
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)
model.fit(X_train, y_train)
test_df = test_df.assign(model_score=model.predict_proba(X_test)[:, 1])

# Fair comparison: eligible test pages only (same pool as ML-07 queue)
eligible_test = test_df[test_df["imp_mar"] >= IMP_FLOOR].copy()
eligible_test = eligible_test.sort_values(
    ["baseline_score", "imp_mar"], ascending=[False, False]
)
y_elig = eligible_test[LABEL].to_numpy()
imp_elig = eligible_test["imp_mar"].to_numpy()  # shared tie-break for rule and model
base_rate = float(y_elig.mean())
baseline_scores = eligible_test["baseline_score"].to_numpy()
model_scores = eligible_test["model_score"].to_numpy()

# Comparison contract: same rows, same client holdout, Precision@K, tie-break=imp_mar
rows = []
for method, scores in [
    ("baseline (ctr_gap)", baseline_scores),
    ("logistic regression", model_scores),
]:
    row = {
        "method": method,
        "split": "client holdout (eligible test pages)",
        "n_test_pages": len(eligible_test),
        "n_test_clients": len(test_clients),
        "base_rate": base_rate,
        "tie_break": "imp_mar",
    }
    for k in (10, 20, 50):
        row[f"precision_at_{k}"] = precision_at_k(
            scores, y_elig, k, tie_break=imp_elig
        )
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("method")

print(f"Train pages: {len(train_df):,}  |  Test pages (all): {len(test_df):,}")
print(f"Test pages (imp_mar >= {IMP_FLOOR}): {len(eligible_test):,}")
print(
    f"Train clients: {train_df['client_hash_id'].nunique():,}  |  "
    f"Test clients: {len(test_clients):,}"
)
print(f"Base rate on eligible test ({LABEL}=1): {base_rate:.3f}\n")
print("Model vs baseline — same eligible test pages:")
print(comparison.round(3).to_string())

OUT = ROOT / "work" / "outputs" / "model_vs_baseline_metrics.json"
CSV_OUT = ROOT / "work" / "outputs" / "model_vs_baseline_comparison.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "label": LABEL,
    "slice": "month=2026-03",
    "population": f"imp_mar>={IMP_FLOOR} test pages",
    "random_state": RANDOM_STATE,
    "n_train_pages": int(len(train_df)),
    "n_test_pages_all": int(len(test_df)),
    "n_test_pages_eligible": int(len(eligible_test)),
    "n_train_clients": int(train_df["client_hash_id"].nunique()),
    "n_test_clients": int(len(test_clients)),
    "methods": comparison.reset_index().to_dict(orient="records"),
}
OUT.write_text(json.dumps(payload, indent=2))
comparison.reset_index().round(3).to_csv(CSV_OUT, index=False)
print(f"\nSaved → {OUT}")
print(f"Saved → {CSV_OUT}")


Train pages: 72,470  |  Test pages (all): 28,971
Test pages (imp_mar >= 500): 17,110
Train clients: 33  |  Test clients: 11
Base rate on eligible test (is_ctr_underperformer=1): 0.464

Model vs baseline — same eligible test pages:
                                                    split  n_test_pages  n_test_clients  base_rate tie_break  precision_at_10  precision_at_20  precision_at_50
method                                                                                                                                                         
baseline (ctr_gap)   client holdout (eligible test pages)         17110              11      0.464   imp_mar              1.0              1.0             1.00
logistic regression  client holdout (eligible test pages)         17110              11      0.464   imp_mar              0.7              0.8             0.84

Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\model_vs_baseline_metrics.json
Saved → c:\Users\rimla

## 4. Errors and interpretation

**Did the model beat the baseline on Precision@K?** No — on eligible held-out test pages (base rate **46.4%**) the fixed ML-07 rule hits **1.0** at K=10/20/50 (by design: `ctr_gap` uses the same ingredients as the label). The model lands **0.7 / 0.8 / 0.84** on the same rows.

**Why the model still matters:** Precision@K is not the whole story. The baseline **sorts zero-click `top_3` pages first** because they share the max `ctr_gap`. The model lifts **high-impression underperformers with non-zero CTR** into the top 10 — pages a reviewer can actually fix (see rank-gap table). Model top-10 can also include **false alarms** (high score, label = 0).

**What the model leans on:** Mostly **`ctr_mar`** and **`position_tier`**. **`imp_mar`** has positive weight — it learns the volume floor the label uses.

**Where the baseline is still weak (even after the imp floor):**
- **Stake ordering:** 100k+ impression underperformers sit at baseline rank ~7,600+ while zero-click pages fill rank 1–50.
- **Actionability:** top baseline picks are often zero-click SERP cases; model top picks mix in weak-but-nonzero CTR pages.
- **Proxy label:** below tier median with `imp_mar >= 500` ≠ “fixing the page will gain clicks” — decision-support only.

**Rank overlap:** **0/10** pages match in baseline vs model top-10 on eligible test — same metric, completely different order.


In [3]:
# --- What the model leans on (coefficients after scaling) ---
prep = model.named_steps["prep"]
clf = model.named_steps["clf"]
coef = pd.Series(clf.coef_[0], index=prep.get_feature_names_out()).sort_values(
    key=np.abs, ascending=False
)
print("Top logistic-regression weights (signed, standardized inputs):")
print(coef.head(8).round(3).to_string())
print()

# --- Baseline ranks on eligible test pages (ML-07 fixed rule) ---
test_ranked = eligible_test.copy()
test_ranked["baseline_rank"] = np.arange(1, len(test_ranked) + 1)
test_ranked["model_rank"] = test_ranked["model_score"].rank(
    ascending=False, method="first"
)

max_gap = test_ranked["baseline_score"].max()
n_tied_max = int((test_ranked["baseline_score"] == max_gap).sum())
print(f"Eligible test pages tied on max baseline_score ({max_gap:.4f}): {n_tied_max:,}")

top10_base = set(test_ranked.head(10)["content_hash_id"])
top10_model = set(test_ranked.nsmallest(10, "model_rank")["content_hash_id"])
print(
    f"Top-10 overlap (same pages in both lists): "
    f"{len(top10_base & top10_model)}/10"
)
print()

# --- Rank disagreements among underperformers ---
under = test_ranked[test_ranked[LABEL] == 1].copy()
under["rank_gap"] = (under["baseline_rank"] - under["model_rank"]).abs()
print("Largest baseline vs model rank gaps (underperformers only, top 5):")
disagree_cols = [
    "content_hash_id",
    "imp_mar",
    "ctr_mar",
    "position_tier",
    "baseline_rank",
    "model_rank",
    "rank_gap",
]
print(
    under.nlargest(5, "rank_gap")[disagree_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}")
)
print()

# --- Baseline top-50: any non-underperformers? (should be 0 on eligible pool) ---
baseline_top50 = test_ranked.head(50)
print(
    f"Non-underperformers in baseline top-50: "
    f"{int((baseline_top50[LABEL] == 0).sum())} (expect 0 on eligible pool)"
)
print()

# --- Model top-10: high-stake picks baseline ranks much lower ---
print("Model top-10 (eligible test) — baseline rank for each:")
model_top10 = test_ranked.nsmallest(10, "model_rank")[
    ["content_hash_id", "imp_mar", "ctr_mar", "position_tier", "baseline_rank", "model_rank", LABEL]
]
print(model_top10.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Top logistic-regression weights (signed, standardized inputs):
cat__position_tier_page_3_5   -5.907
num__ctr_mar                  -3.511
cat__position_tier_deep       -3.019
cat__position_tier_top_3       2.576
cat__position_tier_page_1      1.895
cat__position_tier_striking    1.195
num__pos_avg_mar              -0.735
num__imp_mar                   0.438

Eligible test pages tied on max baseline_score (0.2525): 234
Top-10 overlap (same pages in both lists): 0/10

Largest baseline vs model rank gaps (underperformers only, top 5):
         content_hash_id     imp_mar  ctr_mar position_tier  baseline_rank  model_rank  rank_gap
content_cb6d4179551785be  75296.0000   0.1926        page_1           7852     29.0000 7823.0000
content_59375088eb2eeb27  50236.0000   0.1931        page_1           7867    104.0000 7763.0000
content_b77a49d2e0fe5931  40242.0000   0.1938        page_1           7895    241.0000 7654.0000
content_b99ea6861864dea5 194337.0000   0.1858        page_1           7644 

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.